Data Loading Part:
* In this notebook data loading code is contained.

Importing the MURA dataset from the Drive. (Will do this after the file uploades, in the meantime we will use kaggle dataset).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Installing necessary packages

In [ ]:
!pip install -q torch torchvision  --index-url https://download.pytorch.org/whl/cu121
!pip install -q pandas pillow matplotlib tqdm

Importing Libraries

In [ ]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


import random
import zipfile
from tqdm.auto import tqdm
from PIL import Image, ImageOps
import torch
from torch.utils.data import Dataset
from torchvision import transforms

import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from pathlib import Path

In [ ]:
zip_path = Path("/content/drive/MyDrive/MURA-v1.1.zip")
extract_dir = Path("/content/drive/MyDrive")
mura_dir = extract_dir / "MURA-v1.1"

if mura_dir.is_dir():
    print("Dataset folder already exists, skipping extraction")
else:
  with zipfile.ZipFile(zip_path, "r") as z:
      members = z.infolist()
      total_size = sum(m.file_size for m in members)

      with tqdm(total=total_size, desc="Extracting MURA", unit="B", unit_scale=True, unit_divisor=1024) as pbar:
          for member in members:
              z.extract(member, extract_dir)
              pbar.update(member.file_size)

print("Done")

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive")
MURA_DIR = BASE_DIR / "MURA-v1.1"

print("BASE_DIR exists:", BASE_DIR.exists())
print("MURA_DIR exists:", MURA_DIR.exists())
print("Train exists:", (MURA_DIR / "train").exists())
print("Valid exists:", (MURA_DIR / "valid").exists())
print("Current working dir:", os.getcwd())

In [ ]:
def print_mura_first_n_patients(root, max_patients=20):
    root = Path(root)
    shown = 0

    print(root.name + "/")

    for xr_type in sorted(root.iterdir()):
        if not xr_type.is_dir():
            continue
        print(f"  {xr_type.name}/")

        for patient in sorted(xr_type.iterdir()):
            if not patient.is_dir():
                continue
            if not patient.name.startswith("patient"):
                continue

            shown += 1
            print(f"    {patient.name}/")

            for study in sorted(patient.iterdir()):
                if study.is_dir():
                    print(f"      {study.name}/")
                    imgs = sorted(study.iterdir())
                    for img in imgs[:1]:
                        print(f"        {img.name}")

            if shown >= max_patients:
                return

print_mura_first_n_patients(MURA_DIR / "train", max_patients=20)

A small sanity check to verify we have the correct dataset.

What we understand from the above code output is that there are different levels in our structure.
* ----train/
* --------patientXXXX/
* ------------studyY_label/
* ----------------imageZ.png

In [ ]:
def show_patient_studies(root, patient_ids):
    root = Path(root)

    for patient_id in patient_ids:
        print(f"\n=== {patient_id} ===")
        found = False

        for xr_type in sorted(root.iterdir()):
            if not xr_type.is_dir():
                continue

            patient_path = xr_type / patient_id
            if not patient_path.exists():
                continue

            found = True
            print(f"Found in: {xr_type.name}")

            for study_path in sorted(patient_path.iterdir()):
                if not study_path.is_dir():
                    continue

                images = sorted(study_path.glob("*.png"))

                if not images:
                    print(f"{study_path.name}: no images")
                    continue

                print(f"{study_path.name}: {len(images)} image(s)")

                fig, axes = plt.subplots(1, len(images), figsize=(4 * len(images), 4))
                if len(images) == 1:
                    axes = [axes]

                for ax, img_path in zip(axes, images):
                    img = Image.open(img_path)
                    ax.imshow(img, cmap="gray")
                    ax.set_title(img_path.name)
                    ax.axis("off")

                plt.suptitle(f"{patient_id} - {study_path.name}")
                plt.tight_layout()
                plt.show()

        if not found:
            print("Patient not found")

In [ ]:
patients = ["patient00011","patient00048","patient00148"]
show_patient_studies("/content/drive/MyDrive/MURA-v1.1/train", patients)

The important thing is that per patient we have different studies. Each study is a series of of X-ray images of the same anatomical region. What we want is to aggregate the images of the same study to deduce outcomes.

We will proceed with the training/val split

In [ ]:
TRAIN_CSV = MURA_DIR / "train_labeled_studies.csv"
VALID_CSV = MURA_DIR / "valid_labeled_studies.csv"

After creating the paths for each csv we will verify whether the file exist.

In [ ]:
print("MURA_DIR:", mura_dir)
print("Train CSV exists:", TRAIN_CSV.exists())
print("Valid CSV exists:", VALID_CSV.exists())

assert MURA_DIR.exists(), f"Dataset folder not found: {MURA_DIR}"
assert TRAIN_CSV.exists(), f"Missing file: {TRAIN_CSV}"
assert VALID_CSV.exists(), f"Missing file: {VALID_CSV}"

Inspecting CSVs contents

In [ ]:
train_preview = pd.read_csv(TRAIN_CSV, header=None)
valid_preview = pd.read_csv(VALID_CSV, header=None)

print("First training path:")
print(train_preview.iloc[0, 0])

print("\nFirst validation path:")
print(valid_preview.iloc[0, 0])

In [ ]:
def build_image_dataframe(csv_path, base_dir):
    """
    Reads the MURA study-level CSV and extracts all individual image paths,
    assigning the study label to each image.
    """
    # MURA CSVs don't have headers. Column 0 is path, Column 1 is label.
    df = pd.read_csv(csv_path, header=None, names=['study_path', 'label'])

    image_paths = []
    labels = []

    print(f"Parsing {csv_path.name}...")
    # Wrap with tqdm for a progress bar since iterating disk paths can take a moment
    for _, row in tqdm(df.iterrows(), total=len(df)):
        study_dir = base_dir / row['study_path']

        # Guard against missing directories
        if not study_dir.exists():
            continue

        # Find all pngs in the study directory
        for img_path in study_dir.glob('*.png'):
            image_paths.append(str(img_path))
            labels.append(row['label'])

    img_df = pd.DataFrame({'image_path': image_paths, 'label': labels})
    print(f"Found {len(img_df)} images across {len(df)} studies.\n")
    return img_df

In [ ]:
# Build the dataframes
train_df = build_image_dataframe(TRAIN_CSV, BASE_DIR)
valid_df = build_image_dataframe(VALID_CSV, BASE_DIR)

# Check for class imbalance in the training set
print("Training set label distribution:")
print(train_df['label'].value_counts(normalize=True))



Based on the above outcomes, we have approximately 37k images for 13k studies in the training set. Validation set contains 3k images for 1k studies. The training set distribution shows a slight imbalance where normal cases (0) amount to a 60% of the total.

We have created the corresponding dfs for train and validation, however we have not examined the data enough. As we are discussing images, we find only logical to take a look on some characteristics about size, colour and other traits.

In [ ]:
def inspect_image_properties(df, sample_size=200):
    """
    Inspect a sample of images from a dataframe containing an 'image_path' column.
    Returns a dataframe with image metadata.
    """
    sample_df = df.sample(min(sample_size, len(df)), random_state=42).copy()

    rows = []

    for img_path in tqdm(sample_df["image_path"], total=len(sample_df)):
        img_path = Path(img_path)

        try:
            with Image.open(img_path) as img:
                width, height = img.size
                mode = img.mode  # e.g. L, RGB, RGBA
                n_channels = len(img.getbands())  # e.g. 1 for L, 3 for RGB

                if width > height:
                    orientation = "landscape"
                elif height > width:
                    orientation = "portrait"
                else:
                    orientation = "square"

                rows.append({
                    "image_path": str(img_path),
                    "width": width,
                    "height": height,
                    "aspect_ratio": width / height,
                    "mode": mode,
                    "n_channels": n_channels,
                    "orientation": orientation
                })

        except Exception as e:
            rows.append({
                "image_path": str(img_path),
                "width": None,
                "height": None,
                "aspect_ratio": None,
                "mode": f"ERROR: {e}",
                "n_channels": None,
                "orientation": None
            })

    return pd.DataFrame(rows)

In [ ]:
train_img_info = inspect_image_properties(train_df, sample_size=300)
train_img_info.head()

In [ ]:
print("Modes:")
print(train_img_info["mode"].value_counts(dropna=False))
print()

print("Number of channels:")
print(train_img_info["n_channels"].value_counts(dropna=False))
print()

print("Orientation:")
print(train_img_info["orientation"].value_counts(dropna=False))
print()

print("Width summary:")
print(train_img_info["width"].describe())
print()

print("Height summary:")
print(train_img_info["height"].describe())
print()

print("Aspect ratio summary:")
print(train_img_info["aspect_ratio"].describe())

In [ ]:
size_counts = (
    train_img_info
    .assign(size=lambda x: list(zip(x["width"], x["height"])))
    ["size"]
    .value_counts()
)

print(size_counts.head(20))

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(train_img_info["width"].dropna(), bins=30)
plt.xlabel("Width")
plt.ylabel("Count")
plt.title("Distribution of Image Widths")
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(train_img_info["height"].dropna(), bins=30)
plt.xlabel("Height")
plt.ylabel("Count")
plt.title("Distribution of Image Heights")
plt.show()

In [ ]:
def show_raw_images_with_metadata(info_df, n=6):
    sample = info_df.sample(min(n, len(info_df)), random_state=42).reset_index(drop=True)

    plt.figure(figsize=(15, 8))

    for i, row in sample.iterrows():
        img = Image.open(row["image_path"])

        plt.subplot(2, 3, i + 1)
        if row["n_channels"] == 1:
            plt.imshow(img, cmap="gray")
        else:
            plt.imshow(img)

        plt.title(
            f"{Path(row['image_path']).name}\n"
            f"{row['width']}x{row['height']} | {row['mode']} | {row['orientation']}"
        )
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
show_raw_images_with_metadata(train_img_info, n=6)

It is made clear that images differ from each other. Some are of different size, orientation and even colour.

In [ ]:
def resize_and_pad(img, target_size=224, fill=0):
    img = img.copy()
    img.thumbnail((target_size, target_size))  # preserves aspect ratio

    pad_w = target_size - img.size[0]
    pad_h = target_size - img.size[1]

    padding = (
        pad_w // 2,
        pad_h // 2,
        pad_w - pad_w // 2,
        pad_h - pad_h // 2
    )

    return ImageOps.expand(img, border=padding, fill=fill)

def show_resize_options(image_path, target_size=224):
    img = Image.open(image_path).convert("RGB")

    original = img
    square_resize = img.resize((target_size, target_size))
    padded_resize = resize_and_pad(img, target_size=target_size, fill=0)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(original)
    axes[0].set_title(f"Original\n{original.size[0]} x {original.size[1]}")
    axes[0].axis("off")

    axes[1].imshow(square_resize)
    axes[1].set_title(f"Direct Resize\n{square_resize.size[0]} x {square_resize.size[1]}")
    axes[1].axis("off")

    axes[2].imshow(padded_resize)
    axes[2].set_title(f"Resize + Pad\n{padded_resize.size[0]} x {padded_resize.size[1]}")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
sample_path = train_df.iloc[0]["image_path"]
show_resize_options(sample_path, target_size=224)

In [ ]:
for i in [0, 10, 50]:
    sample_path = train_df.iloc[i]["image_path"]
    print(sample_path)
    show_resize_options(sample_path, target_size=224)

Next step is to create a class that will help us produce an input that is compatible with pytorch.

In [ ]:
class MURADataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        img_path = self.df.iloc[idx]["image_path"]
        label = self.df.iloc[idx]["label"]

        # Keep RGB for pretrained CNNs / ViTs
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(label, dtype=torch.float32)
        return image, label

In [ ]:
class ResizeAndPad:
    def __init__(self, size, fill=0, resample=Image.BILINEAR):
        self.size = size
        self.fill = fill
        self.resample = resample

    def __call__(self, img):
        # Work on a copy to avoid modifying the original PIL image in-place
        img = img.copy()

        w, h = img.size

        # Scale so that the longest side becomes `size`
        scale = self.size / max(w, h)
        new_w = int(round(w * scale))
        new_h = int(round(h * scale))

        img = img.resize((new_w, new_h), self.resample)

        # Compute symmetric padding to reach (size, size)
        pad_left = (self.size - new_w) // 2
        pad_top = (self.size - new_h) // 2
        pad_right = self.size - new_w - pad_left
        pad_bottom = self.size - new_h - pad_top

        img = ImageOps.expand(
            img,
            border=(pad_left, pad_top, pad_right, pad_bottom),
            fill=self.fill
        )

        return img

In [ ]:
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

data_transforms = {
    "train": transforms.Compose([
        ResizeAndPad(224, fill=0),
        transforms.RandomHorizontalFlip(),
        # transforms.RandomRotation(10),
        transforms.ToTensor(),
        normalize
    ]),
    "valid": transforms.Compose([
        ResizeAndPad(224, fill=0),
        transforms.ToTensor(),
        normalize
    ]),
}

In [ ]:
train_dataset = MURADataset(train_df, transform=data_transforms["train"])
valid_dataset = MURADataset(valid_df, transform=data_transforms["valid"])

In [ ]:
image, label = train_dataset[0]
print(image.shape, label)

In [ ]:
def unnormalize_tensor(img_tensor, mean, std):
    """
    Undo normalization so the image can be displayed correctly.
    img_tensor: torch tensor of shape [C, H, W]
    """
    img = img_tensor.clone().cpu()
    for c, (m, s) in enumerate(zip(mean, std)):
        img[c] = img[c] * s + m
    img = img.permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    return img

def show_before_after_samples(df, dataset, n=5, mean=None, std=None, seed=42):
    """
    Shows raw images next to transformed images.

    df: dataframe containing image_path and label
    dataset: MURADataset object with transforms applied
    n: number of samples to display
    """
    if mean is None:
        mean = [0.485, 0.456, 0.406]
    if std is None:
        std = [0.229, 0.224, 0.225]

    random.seed(seed)
    indices = random.sample(range(len(df)), min(n, len(df)))

    fig, axes = plt.subplots(n, 2, figsize=(10, 4 * n))
    if n == 1:
        axes = np.array([axes])

    for row_i, idx in enumerate(indices):
        # Raw image
        img_path = df.iloc[idx]["image_path"]
        label = df.iloc[idx]["label"]
        raw_img = Image.open(img_path).convert("RGB")

        # Transformed image from dataset
        transformed_img, transformed_label = dataset[idx]
        transformed_img_disp = unnormalize_tensor(transformed_img, mean, std)

        # Plot raw
        axes[row_i, 0].imshow(raw_img)
        axes[row_i, 0].set_title(
            f"Before\n{raw_img.size[0]}x{raw_img.size[1]} | label={label}"
        )
        axes[row_i, 0].axis("off")

        # Plot transformed
        axes[row_i, 1].imshow(transformed_img_disp)
        axes[row_i, 1].set_title(
            f"After\n{tuple(transformed_img.shape)} | label={transformed_label.item():.0f}"
        )
        axes[row_i, 1].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
show_before_after_samples(train_df, train_dataset, n=6)

* What the above code cells aim to achieve is to produce a pipeline that is standardized and compatible with deep learning models. We transform the metadata tables (the dfs) into rows with image paths and labels.
* It has to be noted that we also augment the data. Apart from standardization regarding the sizing and the RGB colour which is better option for pretrained models, we also resize, pad and normalize in order to produce better results during training. Normalization does take into account the ImageNet channel normalization as well.


In [ ]:
# Batch size of 32 is a safe starting point for 224x224 images on Colab GPUs
BATCH_SIZE = 32

# num_workers=2 is usually the sweet spot for Colab to avoid IO bottlenecks without crashing
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of validation batches: {len(valid_loader)}")

# Sanity Check: Fetch one batch
train_features, train_labels = next(iter(train_loader))

print(f"\nFeature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels.size()}")
print(f"Feature dtype: {train_features.dtype}")
print(f"Label dtype: {train_labels.dtype}")

# Ensure there are no logical errors in our shapes
assert train_features.dim() == 4, "Images should be a 4D tensor (Batch, Channels, Height, Width)"
assert train_features.size(1) == 3, "Images should have 3 channels (RGB)"
assert train_features.size(2) == 224 and train_features.size(3) == 224, "Images should be 224x224"

The above output confirms that data pipeline is working as expected. We have produced a series of 1151 and 100 train and validation batches respectively while the tensor shape is (batch_size, channels, height, width) = (32, 3, 224, 224). We also use float32 which will help with BCEWithLogitsLoss.